# Experimento 5. El viento y el cambio de objetivo

Cómo se descubrió que el viento limitaba al proyecto, cómo eso cambió el objetivo y qué se encontró después. Cada parte dice de dónde viene:

- **Reproducido con `nox_ml`**: se calcula aquí, con los datos del repositorio.
- **Citado**: viene de los reportes de la investigación (rama `research-archive`: `reports/viento.md`, `reports/RESUMEN.md`, `reports/otras_ciudades.md`) y no se recalcula aquí.

In [ ]:
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import report
from nox_ml.validation import report_experiments as rx

RUNS = report.find_runs()
ROOT = RUNS.parent
inputs = load_inputs(ROOT / "data", "2023-11-01", "2023-12-31")
print(f"{len(inputs.hours)} horas de tarde, {inputs.n_station_rows} observaciones de estación")

## 1. El objetivo original *(citado)*

**Ubicar fuentes individuales de NOₓ con una PINN**: dado el NOₓ de las estaciones y la columna de TEMPO, encontrar *dónde* y *cuánto* emite cada fuente puntual (centrales, aeropuertos, polígonos industriales). El plan de validación lo expresaba como metas de recuperación: un error de ubicación menor a 2 km y un error de emisión menor a 25 % por fuente (`RESUMEN.md`, hallazgo 1 de calidad de datos; `viento.md`, §2). Las PINN de la literatura que ubican una fuente suponen un viento conocido (Chuprov et al., 2025, citado en `otras_ciudades.md`): ese supuesto es el que se pone a prueba en las siguientes secciones.

## 2. Gemelo sintético: ¿qué se pierde con el error del viento? *(reproducido)*

Se simulan observaciones con emisiones conocidas y con el **viento verdadero**, y se invierten con el viento al que se le añade un error aleatorio de dirección en cada hora, de 5°, 10°, 20° y 30° (desviación estándar). Con 0° el modelo es perfecto, de modo que ese caso mide el piso del método.

Diseño del gemelo (`rx.wind_error_twin`): las emisiones verdaderas son las a priori por un factor lognormal al azar (puntuales: dispersión 0.5; regiones viales: 0.3); las observaciones son la respuesta del modelo de la línea base, un fondo horario de 5 ± 3 ppb y ruido de 8 ppb en estaciones y de 2×10¹⁵ moléculas/cm² en TEMPO. 3 semillas (emisiones y ruido distintos); el error del viento es independiente en cada hora. Tarda unos 2.5 minutos: cada error de viento reconstruye las matrices. Una advertencia: una desviación estándar de 30° corresponde a un error **mediano** de unos 20°; el error mediano medido en la sección siguiente (30°) equivale a una desviación estándar de unos 45°, más que lo que se ve aquí.

In [ ]:
twin = rx.wind_error_twin(inputs, sigmas=(0, 5, 10, 20, 30), seeds=(0, 1, 2))
rx.twin_summary(twin).round(1)

In [ ]:
rx.plot_wind_twin(twin)

Error relativo mediano (%) de cada tipo de incógnita según el error del viento (σ, grados). «punto» es cada una de las 4 fuentes puntuales; «zona», cada una de las 9 regiones viales en días hábiles; «total vial», la suma de las regiones.

**Lectura.**

- **Las fuentes puntuales se pierden.** Su error mediano es de 1 % sin error de viento, 7 % con 5°, **26 % con 10°** (ya por encima de la meta de 25 %), 72 % con 20° y **119 % con 30°**. Se pierden primero las de menor emisión a priori (Nonoalco, 277 t/año, y los aeropuertos, 500 t/año: 350 %, 240 % y 72 % con 30°); solo la central de Valle de México, de 2,615 t/año, aguanta (18 %).
- **Lo agregado sobrevive.** El total vial pasa de 0.2 % a 13 % con 30°, y el cociente domingo ÷ hábil se equivoca menos de 7 %. Las 9 regiones por separado se degradan poco a poco (22 % mediano con 30°; la peor, cerca de 95 %); el total se equivoca menos que cada región porque parte de sus errores se compensan al sumar.
- Es lo que la investigación reportó con otro gemelo *(citado, `viento.md`, §2)*:

| Error de dirección (1σ) | Vial por región: mediana (máx) | Acolman / AIFA / AICM |
|---|---|---|
| ±10° | 3 % (9 %) | 7 / 10 / 14 % |
| ±20° | 9 % (24 %) | 18 / 27 / 22 % |
| ±30° | 15 % (36 %) | 32 / 35 / 37 % |

Los números no coinciden porque el gemelo y el ruido no son los mismos, pero la forma sí: con ±30° las fuentes grandes ya no cumplen la meta y la emisión vial sí.

## 3. El error real del viento *(reproducido)*

¿Qué error de dirección tiene el viento que se usa de verdad? Se predice el viento de cada estación SIMAT con las demás del mismo instante (se oculta una a la vez), de 12 a 17 h locales en noviembre–diciembre de 2023, solo con viento observado ≥ 1 m/s. Se prueban tres predictores: el **vector medio** de las demás estaciones (lo que usa la inversión), la **interpolación IDW** de las demás y **ERA5** (una celda de 0.25° sobre la ciudad).

In [ ]:
winds, station_xy = rx.station_winds(ROOT / "data", "2023-11-01", "2023-12-31")
errors = rx.wind_leave_one_out(winds, station_xy)
era5_errors = rx.era5_wind_errors(rx.era5_wind(ROOT / "data"), errors)
rx.wind_error_summary(errors, era5_errors).round(1)

In [ ]:
rx.plot_wind_errors(errors, era5_errors)

**Lectura.**

- **Mediana de 32° con el vector medio, 29° con IDW y 56° con ERA5.** La investigación midió 32°, 28° y 55° con 22 estaciones y 6,857 estaciones-hora; aquí son 20 estaciones y 6,202 estaciones-hora (las que tienen viento en este periodo en el catálogo del repositorio) y el resultado es el mismo. Un error mediano de 30° equivale a una desviación estándar de unos 45°.
- **Un viento de 1–3 km mejora a ERA5 pero no baja de ~20°.** Las estaciones con más error son las mal expuestas (CHO, FAC, ACO, CUA, 55–127°) y las mejores, las del centro y el oriente plano (22–25°). La investigación midió además que estaciones a menos de 10 km ya difieren 32–38°: ese piso es ruido local de cada sitio, y ningún campo de viento bajará de unos 20–25° de error *(citado, `viento.md`, §1)*.
- Comparado con la sección anterior: el error real (mediana de 30°) está **más allá de lo que el gemelo con 30° de desviación estándar** ya muestra como pérdida de las fuentes individuales.

## 4. La decisión *(citado)*

Con ese error de viento (`viento.md`, `RESUMEN.md`, «Revisión de conclusiones») la investigación concluyó que:

- **Las fuentes puntuales individuales no son recuperables** con el viento disponible: con ±30° ni las fuentes grandes cumplen la meta (32–37 %) y de 12 clústeres industriales solo 1 queda bajo 25 %.
- **Sí son confiables** la emisión vial **por zona** y su ciclo semanal (domingo ≈ 0.4–0.5 del día hábil), y los **cambios de emisión** con el estimador normalizado por meteorología (domingo, confinamiento de 2020 con −41/−42 % con la configuración estándar (−35 % fue el valor de una configuración anterior de la investigación), contingencias).
- **El objetivo cambió** de «ubicar fuentes individuales» a **estimar la emisión por zona y sus cambios** separando el efecto de la emisión del efecto del clima. Es el objetivo de este repositorio (`README.md`) y la razón de que los cuadernos 2 y 5 usen el cociente domingo ÷ hábil y no la posición de una fuente.
- La prioridad pasó a ser **mejorar el viento**: pedir a SEDEMA su WRF de 1 km o correr uno propio (`viento.md`, §3–4).

## 5. El giro posterior: el viento ya no es lo que limita

### Los Ángeles *(citado, `otras_ciudades.md`)*

Para probar «mejor viento, mejor inversión» con datos reales, la investigación repitió el método en Los Ángeles, donde el viento es mucho mejor: el error de dirección es de 16–17° (campo HRRR de 3 km o IDW de estaciones) contra 30° del vector medio. Mismas 363 horas de noviembre–diciembre de 2023:

| Métrica | Pluma, viento medio (30°) | Pluma, HRRR (16°) | **Puffs**, viento medio (30°) | Puffs, HRRR (16°) |
|---|---|---|---|---|
| Error con estación oculta (ppb) | 13.6 | 14.1 | **13.3** | 13.5 |
| Correlación del modelo con TEMPO | 0.24 | 0.14 | **0.49** | 0.47 |
| Vial en días hábiles (t/año; a priori 40,300) | 8,200 | −1,000 | **36,000** | 34,100 |
| Placebo (± medio ancho) | ±0.17 | ±0.19 | **±0.14** | ±0.16 |

**Un viento mucho mejor no mejoró ninguna métrica** (el error subió de 13.6 a 14.1 ppb). **Pasar de pluma estacionaria a puffs sí**: con el mismo viento uniforme duplicó la correlación con TEMPO, devolvió una emisión vial físicamente razonable y estrechó el placebo, y con puffs el viento HRRR no agregó nada. En la CDMX, una prueba previa con puffs subió la correlación con TEMPO de 0.74 a 0.83 pero **empeoró** el error en estaciones (14.1 → 15.1 ppb), y el reporte pide revisarlo antes de adoptarlos. Conclusión citada: «La prioridad es el transporte dependiente del tiempo […], no un viento más fino».

### En este repositorio *(reproducido)*

Dos cambios, comparados con la misma métrica (error con estación oculta, ppb; negativo es mejor):

1. **Girar el viento** 15° y 30° (ablación de `nox-ml diagnose`, cuaderno 4): un error sistemático de dirección.
2. **Cambiar la pluma por puffs** con la PINN: `pinn_plume` frente a `pinn_puff`. Estas corridas existen en la máquina del dueño del proyecto; si no están en `runs/`, se usan los valores que el proyecto reportó (12.49 y 12.08 ppb) y la tabla lo dice en la columna `fuente`.

In [ ]:
diagnostics = rx.diagnostics_for(ROOT / "configs" / "experiments" / "baseline.yaml", RUNS)
plume_puff = rx.plume_to_puff(RUNS)
plume_puff

In [ ]:
changes = rx.error_changes(diagnostics["ablation"], plume_puff)
changes.round(2)

In [ ]:
rx.plot_error_changes(changes)

**Lectura.**

- Girar el viento 30° **no empeora** el error (13.6 → 13.3 ppb) ni la correlación con TEMPO (0.74) ni el cociente domingo ÷ hábil (0.42 → 0.40). Un error sistemático de dirección que sería grande en la sección 3 **no se nota** en estas métricas.
- Pasar de pluma a puffs con la PINN lo baja de 12.49 a 12.08 ppb (3 %).
- **Los tres cambios caen dentro de la banda gris**: el error con estación oculta cambia ±0.5 ppb solo por qué estaciones se usen, así que en este ppb no se distingue entre girar el viento y cambiar de transporte. Lo que sí muestra es que **girar el viento no daña**, es decir, que la dirección ya no es lo que limita a este modelo. La evidencia a favor de los puffs no está en esta barra: está en Los Ángeles, donde la mejora es grande (correlación con TEMPO de 0.24 a 0.49), y en la correlación con TEMPO de la CDMX (0.74 a 0.83), ambas citadas arriba.

### Cómo queda la historia

1. Se quería ubicar fuentes individuales con una PINN.
2. Con un gemelo se vio que un error de viento realista **las pierde** y deja lo agregado.
3. El error real del viento (30°) lo confirmó.
4. El objetivo pasó a **emisión por zona y cambios de emisión**, y el viento a ser la prioridad.
5. Una prueba con un viento mucho mejor (Los Ángeles) **no ayudó**, y el transporte con memoria (puffs) sí. La prioridad pasó del viento al **transporte**, y `pinn_puff` (ADR 0005) combina la PINN con ese transporte.